In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH = "Project_Report[1].pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the pdf")
print("\n .... 431 page preview (first 500 character):")
print(pages[35].page_content[:500])

Loaded 52 pages from the pdf

 .... 431 page preview (first 500 character):
Chapter 5. Analysis and Discussion  
 
score. From figure 5.1, it can be seen that Random Forest Regressor is the best per- former 
with approximately 88 percent accuracy score compared to other methods, and Simple Linear 
Regressor is the poor performer with an accuracy score of 81.2 percent. 
 
 
5.2 Discussion 
RQ1: What are the critical features that influence product sales? 
 
Answer: 
It was clearly observed in figure 4.13 that the price of the products and followed by  
the type of outlet


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators = ["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

len(chunks)

154

In [6]:
chunks[100].page_content

'Chapter 4.  Results  \n \n \nFigure 4.12: Accuracy Generated by K-Nearest Neighbor \n \n \n \n4.4 Random Forest Regressor \nThe Values present in figure 4.13 are generated by Random Forest Regressor In figure \n4.14 , Real Values and Predicted Values are generated by Random Forest Regressor , \nfigure 4.15 , represents the Graph on the basis of Real values and Predicted values , figure \n4.16 shows the accuracy of Random Forest Regressor \n \n \nThe outcomes that Random Forest Regressor obtains are as follows: \n \n \n \nFigure 4.13: Values generated by Random Forest Regressor'

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks , embeddings)

print(f"vector store is ready.  {vector_store._collection.count()} vectors stored")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3931.97it/s]


vector store is ready.  308 vectors stored


In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 3})
test_query = "What is machine learning algorithms , explain breif about this."
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved , 1):
    print(f"...... chunks{i}.....")
    print(doc.page_content[:500])
    print()

...... chunks1.....
4 Chapter 1.  Introduction 
 
 
 
 
Figure 1.1: Types of Machine Learning 
 
 
experience. In general, Machine Learning is a program that can manage various  tasks by 
analyzing and exploring data. 
Common Machine Learning applications such as email spam detection, credit card 
fraud, stock predictions, smart assistants, product recommendations, self-driving cars, 
sentiment analysis, etc. 
 
Supervised  Learning:  
The most popular model for performing Machine Learning processes is supervised  

...... chunks2.....
4 Chapter 1.  Introduction 
 
 
 
 
Figure 1.1: Types of Machine Learning 
 
 
experience. In general, Machine Learning is a program that can manage various  tasks by 
analyzing and exploring data. 
Common Machine Learning applications such as email spam detection, credit card 
fraud, stock predictions, smart assistants, product recommendations, self-driving cars, 
sentiment analysis, etc. 
 
Supervised  Learning:  
The most popular model for performing 

In [ ]:
## RAG Pipeline

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

SYSTEM_PROMPT = """\
You are a helpful machine learning assistant.
Answer the question using only the context provided below.
If the context does not contain enough information, say so clearly.

Context : {context}
"""

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

    
prompt = ChatPromptTemplate.from_messages([
    ("system" , SYSTEM_PROMPT),
    ("human" , "{question}"),
])

llm = ChatGroq(
    model = "llama-3.1-8b-instant",
    temperature = 0,
)

chain  =(
    {"context" : retriever | format_docs , "question" : RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

print("RAG chain is assembled.")

RAG chain is assembled.


In [22]:
question = "What is heat map and how it is useful in data correlation."
print(f"Q: {question}\n")
print("A:" , chain.invoke(question))

Q: What is heat map and how it is useful in data correlation.

A: A heat map is a graphical representation of data where values are depicted by color. It is a two-dimensional representation of data where values are mapped to colors, with the color intensity or hue indicating the magnitude of the value. In the context of data correlation, a heat map is particularly useful for visualizing the correlation between different variables.

In a heat map, the correlation between variables is typically represented by a color scale, where:

- High positive correlation is represented by a bright color (e.g., red or yellow)
- Low positive correlation is represented by a light color (e.g., green or blue)
- High negative correlation is represented by a dark color (e.g., blue or purple)
- Low negative correlation is represented by a light color (e.g., green or yellow)

Heat maps are useful in data correlation for several reasons:

1. **Visual representation**: Heat maps provide a clear and intuitive v